# Phase 7 — Cross-Domain Generalization

**Objective**: Evaluate U-Net (ResNet34) and SegFormer-B1 **zero-shot** on a new dataset outside the BraTS-PEDs training distribution (domain shift test). No retraining or fine-tuning is performed.

**Workflow**:
1. Load both trained checkpoints
2. Point `NEW_DATASET_PATH` to a preprocessed split of the new dataset
3. Run the full 3D evaluation pipeline (predict → post-process → Dice + HD95)
4. Compare against in-domain results → report performance drop

**Prerequisites**:
- The new dataset must be preprocessed into the same `.npy` format as the pediatric data.
  Run `02_preprocessing.ipynb` pointing `DATA_ROOT` at the new dataset directory.
  Expected structure:
  ```
  NEW_DATASET_PATH/
      images/  <subject_id>_slice<idx>.npy   # [4, H, W] float32
      masks/   <subject_id>_slice<idx>.npy   # [H, W]    int8
  ```
- Label convention must be `{0=BG, 1=NCR, 2=ED, 3=ET}` (apply `4→3` remap in preprocessing).
- Voxel spacing must be 1 mm isotropic (BraTS standard; adjust `VOXEL_SPACING` otherwise).

---

**In-domain reference results** (test set, best checkpoints):

| Model | NCR | ED | ET | fg Dice |
|---|---|---|---|---|
| U-Net/ResNet34 | 0.8060 | 0.8111 | 0.8202 | 0.8124 |
| SegFormer-B1 | 0.8580 | 0.8008 | 0.8598 | 0.8396 |

## 1 · Imports

In [ ]:
import os
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import segmentation_models_pytorch as smp

# Project source
sys.path.insert(0, str(Path(".").resolve()))
from src.eval_utils import (
    get_subject_ids,
    infer_dataset_shape,
    predict_volume,
    remove_small_components,
    compute_dice_volume,
    compute_hd95_volume,
)
from src.models import get_segformer
from src.train_utils import load_checkpoint

print(f"PyTorch      : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
print(f"SMP          : {smp.__version__}")

## 2 · Configuration

In [ ]:
# ---------------------------------------------------------------------------
# Paths — orignal in-domain checkpoints
# ---------------------------------------------------------------------------
UNET_CKPT      = Path("checkpoints/unet/best.pth")
SEGFORMER_CKPT = Path("checkpoints/segformer/best.pth")

# In-domain test split (BraTS-PEDs) — used as reference to confirm eval parity
INDOMAIN_PATH  = Path("processed_dataset/test")

# ---------------------------------------------------------------------------
# NEW DATASET — point this to the preprocessed cross-domain split.
# Replace the placeholder path before running the cross-domain section.
# ---------------------------------------------------------------------------
NEW_DATASET_PATH = Path("data/processed_adult_brats/test")   # ← PLACEHOLDER

# ---------------------------------------------------------------------------
# In-domain reference results (from Phase 3 & 5 test evaluations)
# ---------------------------------------------------------------------------
INDOMAIN_REF = {
    "unet": {
        "dice_NCR": 0.8060, "dice_ED": 0.8111, "dice_ET": 0.8202,
        "mean_fg_dice": 0.8124,
        "hd95_NCR": 10.70, "hd95_ED": 7.22, "hd95_ET": 16.19,
    },
    "segformer": {
        "dice_NCR": 0.8580, "dice_ED": 0.8008, "dice_ET": 0.8598,
        "mean_fg_dice": 0.8396,
        "hd95_NCR": float("nan"), "hd95_ED": float("nan"), "hd95_ET": float("nan"),
    },
}

# ---------------------------------------------------------------------------
# Evaluation hyperparameters
# ---------------------------------------------------------------------------
DEVICE               = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CROP_SIZE            = 192        # must match training
MIN_COMPONENT_VOXELS = 50         # post-processing threshold
VOXEL_SPACING        = (1.0, 1.0, 1.0)  # mm; BraTS standard isotropic
N_WORST              = 3          # worst subjects to visualise per model
BB_PAD               = 20         # bounding-box padding (px) in visualisation

# Output directory for figures
OUT_DIR = Path("training_outputs")
OUT_DIR.mkdir(exist_ok=True)

print(f"Device       : {DEVICE}")
print(f"Crop size    : {CROP_SIZE}×{CROP_SIZE}")
print(f"U-Net ckpt   : {UNET_CKPT}  (exists={UNET_CKPT.exists()})")
print(f"SegFormer ckpt: {SEGFORMER_CKPT}  (exists={SEGFORMER_CKPT.exists()})")
print(f"New dataset  : {NEW_DATASET_PATH}  (exists={NEW_DATASET_PATH.exists()})")

## 3 · Load Models

In [ ]:
# ── U-Net / ResNet34 ────────────────────────────────────────────────────────
unet = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,    # weights loaded from checkpoint below
    in_channels=4,
    classes=4,
    activation=None,
).to(DEVICE)

unet_meta = load_checkpoint(str(UNET_CKPT), unet, device=DEVICE)
unet.eval()

print(f"U-Net loaded  — epoch {unet_meta['epoch']}, "
      f"val fg Dice = {unet_meta['metrics'].get('fg_dice', 'N/A'):.4f}")

# ── SegFormer-B1 ────────────────────────────────────────────────────────────
segformer = get_segformer(model_checkpoint="nvidia/mit-b1", num_classes=4).to(DEVICE)

segformer_meta = load_checkpoint(str(SEGFORMER_CKPT), segformer, device=DEVICE)
segformer.eval()

print(f"SegFormer loaded — epoch {segformer_meta['epoch']}, "
      f"val fg Dice = {segformer_meta['metrics'].get('fg_dice', 'N/A'):.4f}")

## 4 · Evaluation Helper

Single function that runs the full 3D evaluation loop (predict → post-process → Dice + HD95) on any preprocessed split directory and returns a per-subject DataFrame.

In [ ]:
def evaluate_split(
    model: torch.nn.Module,
    data_dir,
    model_name: str = "model",
    voxel_spacing=VOXEL_SPACING,
    min_component_voxels: int = MIN_COMPONENT_VOXELS,
    crop_size: int = CROP_SIZE,
    device=DEVICE,
    verbose: bool = True,
) -> pd.DataFrame:
    """Run 3D Dice + HD95 evaluation on all subjects in a preprocessed split.

    Works on any dataset following the ``<subject_id>_slice<idx>.npy`` naming
    convention.  Spatial dimensions (orig_size, n_slices) are auto-detected
    per subject via ``infer_dataset_shape()``.

    Returns a DataFrame with one row per subject and columns:
        subject_id, dice_background, dice_NCR, dice_ED, dice_ET,
        mean_fg_dice, hd95_NCR, hd95_ED, hd95_ET
    """
    data_dir = Path(data_dir)
    subject_ids = get_subject_ids(str(data_dir))
    if not subject_ids:
        raise RuntimeError(f"No subjects found in {data_dir}")

    if verbose:
        print(f"[{model_name}] Evaluating {len(subject_ids)} subjects in {data_dir}")

    records = []
    warning_log = []

    model.eval()

    for i, sid in enumerate(subject_ids, 1):
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")

            # 3D inference — auto-detects orig_size & n_slices
            pred_vol, gt_vol = predict_volume(
                model, str(data_dir), sid, device, crop_size=crop_size
            )

            # Post-processing: remove confetti
            pred_vol = remove_small_components(pred_vol, min_voxels=min_component_voxels)

            # Metrics
            dice = compute_dice_volume(pred_vol, gt_vol)
            hd95 = compute_hd95_volume(pred_vol, gt_vol, voxel_spacing=voxel_spacing)

            for w in caught:
                warning_log.append(f"  {sid}: {w.category.__name__}: {w.message}")

        mean_fg = float(np.mean([dice["dice_NCR"], dice["dice_ED"], dice["dice_ET"]]))
        records.append({
            "subject_id": sid,
            "dice_background": dice["dice_background"],
            "dice_NCR": dice["dice_NCR"],
            "dice_ED": dice["dice_ED"],
            "dice_ET": dice["dice_ET"],
            "mean_fg_dice": mean_fg,
            "hd95_NCR": hd95["hd95_NCR"],
            "hd95_ED": hd95["hd95_ED"],
            "hd95_ET": hd95["hd95_ET"],
        })

        if verbose and i % max(1, len(subject_ids) // 5) == 0:
            print(f"  [{i:3d}/{len(subject_ids)}] {sid}  "
                  f"fg_Dice={mean_fg:.4f}")

    df = pd.DataFrame(records)

    if verbose:
        print(f"\n  Done. {len(warning_log)} HD95 edge-case warnings.")
        for msg in warning_log:
            print(msg)

    return df


print("evaluate_split() defined.")

## 5 · Cross-Domain Dataset Check

Inspect the new dataset before running inference — confirm it is accessible and correctly structured.

In [ ]:
NEW_DATASET_AVAILABLE = NEW_DATASET_PATH.exists()

if not NEW_DATASET_AVAILABLE:
    print("⚠  NEW_DATASET_PATH does not exist yet — cross-domain cells will be skipped.")
    print(f"   Set NEW_DATASET_PATH in Cell 2 to a valid preprocessed split directory.")
    print("   Expected structure:")
    print("     NEW_DATASET_PATH/")
    print("         images/  <subject_id>_slice<idx>.npy   # [4, H, W] float32")
    print("         masks/   <subject_id>_slice<idx>.npy   # [H, W]    int8")
else:
    new_ids = get_subject_ids(str(NEW_DATASET_PATH))
    sample_orig, sample_n = infer_dataset_shape(str(NEW_DATASET_PATH), new_ids[0])
    print(f"✅  New dataset found: {len(new_ids)} subjects")
    print(f"   Auto-detected shape: {sample_orig}×{sample_orig}×{sample_n}")
    print(f"   First 5 subjects: {new_ids[:5]}")

## 6 · Cross-Domain Evaluation — U-Net

Run full 3D inference on the new dataset with the U-Net model.

In [ ]:
if not NEW_DATASET_AVAILABLE:
    print("Skipped — new dataset not available.")
    df_unet_cross = None
else:
    df_unet_cross = evaluate_split(
        unet, NEW_DATASET_PATH, model_name="U-Net", verbose=True
    )

## 7 · Cross-Domain Evaluation — SegFormer

In [ ]:
if not NEW_DATASET_AVAILABLE:
    print("Skipped — new dataset not available.")
    df_segformer_cross = None
else:
    df_segformer_cross = evaluate_split(
        segformer, NEW_DATASET_PATH, model_name="SegFormer", verbose=True
    )

## 8 · Aggregate Summary & Comparison Table

Structure: **[Model] | [In-Domain Dice/HD95] | [Cross-Domain Dice/HD95] | [Drop %]**

In [ ]:
def _agg(df: pd.DataFrame) -> dict:
    """Compute mean over subjects for key metrics."""
    return {
        "dice_NCR":     float(df["dice_NCR"].mean()),
        "dice_ED":      float(df["dice_ED"].mean()),
        "dice_ET":      float(df["dice_ET"].mean()),
        "mean_fg_dice": float(df["mean_fg_dice"].mean()),
        "hd95_NCR":     float(np.nanmean(df["hd95_NCR"])),
        "hd95_ED":      float(np.nanmean(df["hd95_ED"])),
        "hd95_ET":      float(np.nanmean(df["hd95_ET"])),
    }


def _drop_pct(indomain_val, cross_val, higher_is_better=True):
    """Return signed performance change as a percentage string."""
    if np.isnan(indomain_val) or np.isnan(cross_val):
        return "N/A"
    if indomain_val == 0:
        return "N/A"
    change = (cross_val - indomain_val) / abs(indomain_val) * 100.0
    if not higher_is_better:
        change = -change   # for HD95, lower is better → flip sign
    sign = "+" if change >= 0 else ""
    return f"{sign}{change:.1f}%"


# ── Build summary rows ──────────────────────────────────────────────────────
rows = []
for model_key, df_cross in [("unet", df_unet_cross), ("segformer", df_segformer_cross)]:
    ref = INDOMAIN_REF[model_key]
    label = "U-Net/ResNet34" if model_key == "unet" else "SegFormer-B1"

    if df_cross is None:
        for metric in ["dice_NCR", "dice_ED", "dice_ET", "mean_fg_dice",
                       "hd95_NCR", "hd95_ED", "hd95_ET"]:
            rows.append({
                "Model": label, "Metric": metric,
                "In-Domain": f"{ref[metric]:.4f}" if not np.isnan(ref[metric]) else "N/A",
                "Cross-Domain": "not run",
                "Change": "N/A",
            })
    else:
        cross = _agg(df_cross)
        for metric in ["dice_NCR", "dice_ED", "dice_ET", "mean_fg_dice"]:
            rows.append({
                "Model": label, "Metric": metric,
                "In-Domain": f"{ref[metric]:.4f}",
                "Cross-Domain": f"{cross[metric]:.4f}",
                "Change": _drop_pct(ref[metric], cross[metric], higher_is_better=True),
            })
        for metric in ["hd95_NCR", "hd95_ED", "hd95_ET"]:
            iv = ref[metric]
            cv = cross[metric]
            rows.append({
                "Model": label, "Metric": metric,
                "In-Domain": f"{iv:.2f} mm" if not np.isnan(iv) else "N/A",
                "Cross-Domain": f"{cv:.2f} mm" if not np.isnan(cv) else "N/A",
                # For HD95, lower is better → negative change = improvement
                "Change": _drop_pct(iv, cv, higher_is_better=False),
            })

summary_df = pd.DataFrame(rows)

print("=" * 72)
print("CROSS-DOMAIN GENERALIZATION SUMMARY")
print("=" * 72)
print(summary_df.to_string(index=False))

## 9 · Per-Subject Distribution Plot

Box-plots of fg Dice across subjects — in-domain (filled) vs cross-domain (hatched).

In [ ]:
if df_unet_cross is None and df_segformer_cross is None:
    print("Skipped — no cross-domain data available.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    fig.suptitle("Per-Class Dice: In-Domain vs Cross-Domain", fontsize=13, fontweight="bold")

    classes = ["dice_NCR", "dice_ED", "dice_ET"]
    titles  = ["NCR (Necrotic Core)", "ED (Edema)", "ET (Enhancing Tumor)"]
    colors  = {"unet": "#4472C4", "segformer": "#ED7D31"}

    for ax, col, title in zip(axes, classes, titles):
        plot_data   = []
        plot_labels = []
        box_colors  = []

        # In-domain reference: use scalar ref value as a single-point box
        for model_key, ref_label, color in [
            ("unet", "U-Net\nIn-Domain", colors["unet"]),
            ("segformer", "SegFormer\nIn-Domain", colors["segformer"]),
        ]:
            plot_data.append([INDOMAIN_REF[model_key][col]])
            plot_labels.append(ref_label)
            box_colors.append(color)

        # Cross-domain distributions
        for df_cross, ref_label, color in [
            (df_unet_cross, "U-Net\nCross-Domain", colors["unet"]),
            (df_segformer_cross, "SegFormer\nCross-Domain", colors["segformer"]),
        ]:
            if df_cross is not None:
                plot_data.append(df_cross[col].values)
                plot_labels.append(ref_label)
                box_colors.append(color)

        bp = ax.boxplot(
            plot_data, labels=plot_labels, patch_artist=True,
            medianprops=dict(color="black", linewidth=2),
        )
        for patch, color in zip(bp["boxes"], box_colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)

        ax.set_title(title, fontsize=11)
        ax.set_ylabel("Dice coefficient")
        ax.set_ylim(0, 1.05)
        ax.grid(axis="y", linestyle="--", alpha=0.5)
        ax.tick_params(axis="x", labelsize=8)

    plt.tight_layout()
    out_path = OUT_DIR / "cross_domain_dice_distribution.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {out_path}")

## 10 · Worst-Case Failure Analysis (Cross-Domain)

Visualise the N_WORST subjects with the lowest fg Dice on the cross-domain dataset for each model.
Panel: T1c (channel 0) · GT overlay · Prediction overlay — zoomed to tumour bounding box.

In [ ]:
_LABEL_COLORS = np.array([
    [0.0, 0.0, 0.0, 0.0],   # 0 = BG: transparent
    [1.0, 0.2, 0.2, 0.5],   # 1 = NCR: red
    [0.2, 0.9, 0.2, 0.5],   # 2 = ED:  green
    [1.0, 1.0, 0.0, 0.5],   # 3 = ET:  yellow
], dtype=np.float32)


def _mask_to_rgba(mask_2d: np.ndarray) -> np.ndarray:
    h, w = mask_2d.shape
    rgba = np.zeros((h, w, 4), dtype=np.float32)
    for c in range(len(_LABEL_COLORS)):
        rgba[mask_2d == c] = _LABEL_COLORS[c]
    return rgba


def _tumour_bbox_2d(gt2d: np.ndarray, pred2d: np.ndarray, pad: int = BB_PAD):
    fg = ((gt2d > 0) | (pred2d > 0))
    if not fg.any():
        s = gt2d.shape[0]
        return 0, s, 0, s
    rows = np.where(fg.any(axis=1))[0]
    cols = np.where(fg.any(axis=0))[0]
    r0, r1 = max(rows[0] - pad, 0), min(rows[-1] + pad + 1, gt2d.shape[0])
    c0, c1 = max(cols[0] - pad, 0), min(cols[-1] + pad + 1, gt2d.shape[1])
    return r0, r1, c0, c1


def _best_gt_slice(gt_vol: np.ndarray) -> int:
    et_counts = (gt_vol == 3).sum(axis=(0, 1))
    if et_counts.max() > 0:
        return int(et_counts.argmax())
    fg_counts = (gt_vol > 0).sum(axis=(0, 1))
    return int(fg_counts.argmax())


def plot_worst_subjects(model, model_name: str, df_cross: pd.DataFrame, n_worst: int = N_WORST):
    """Re-infer and plot the n_worst subjects by fg Dice on the cross-domain set."""
    worst = df_cross.nsmallest(n_worst, "mean_fg_dice")

    for rank, (_, row) in enumerate(worst.iterrows(), 1):
        sid = row["subject_id"]
        pred_vol, gt_vol = predict_volume(model, str(NEW_DATASET_PATH), sid, DEVICE)
        pred_vol = remove_small_components(pred_vol)

        sl = _best_gt_slice(gt_vol)
        gt2d   = gt_vol[:, :, sl]
        pred2d = pred_vol[:, :, sl]

        # Load T1c channel from .npy for visualisation
        slice_file = next(
            f for f in os.listdir(os.path.join(str(NEW_DATASET_PATH), "images"))
            if f.startswith(sid + "_slice") and int(f.split("_slice")[1].replace(".npy", "")) == sl
        )
        t1c = np.load(os.path.join(str(NEW_DATASET_PATH), "images", slice_file))[0]

        r0, r1, c0, c1 = _tumour_bbox_2d(gt2d, pred2d)
        t1c_crop  = t1c[r0:r1, c0:c1]
        gt_crop   = gt2d[r0:r1, c0:c1]
        pred_crop = pred2d[r0:r1, c0:c1]

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        fig.suptitle(
            f"[{model_name}] Rank {rank} worst — {sid}\n"
            f"slice={sl}  ET Dice={row['dice_ET']:.3f}  "
            f"ET HD95={row['hd95_ET']:.1f if not np.isnan(row['hd95_ET']) else 'NaN'} mm",
            fontsize=10,
        )

        axes[0].imshow(t1c_crop, cmap="gray")
        axes[0].set_title("T1c")
        axes[0].axis("off")

        axes[1].imshow(t1c_crop, cmap="gray")
        axes[1].imshow(_mask_to_rgba(gt_crop))
        axes[1].set_title("Ground Truth")
        axes[1].axis("off")

        axes[2].imshow(t1c_crop, cmap="gray")
        axes[2].imshow(_mask_to_rgba(pred_crop))
        axes[2].set_title("Prediction")
        axes[2].axis("off")

        legend_patches = [
            mpatches.Patch(color="red",    alpha=0.6, label="NCR"),
            mpatches.Patch(color="green",  alpha=0.6, label="ED"),
            mpatches.Patch(color="yellow", alpha=0.6, label="ET"),
        ]
        axes[2].legend(handles=legend_patches, loc="lower right", fontsize=8)

        plt.tight_layout()
        safe_model = model_name.lower().replace("/", "_").replace("-", "")
        out = OUT_DIR / f"cross_domain_failure_{safe_model}_rank{rank:02d}_{sid}.png"
        plt.savefig(out, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"  Saved → {out}")


if not NEW_DATASET_AVAILABLE:
    print("Skipped — new dataset not available.")
else:
    print("--- U-Net worst subjects ---")
    plot_worst_subjects(unet, "U-Net", df_unet_cross)
    print("--- SegFormer worst subjects ---")
    plot_worst_subjects(segformer, "SegFormer", df_segformer_cross)

## 11 · Per-Subject Results Table (Cross-Domain)

In [ ]:
if df_unet_cross is not None:
    print("U-Net — cross-domain results (sorted by mean_fg_dice ascending)")
    display_cols = ["subject_id", "dice_NCR", "dice_ED", "dice_ET",
                    "mean_fg_dice", "hd95_NCR", "hd95_ED", "hd95_ET"]
    print(df_unet_cross[display_cols].sort_values("mean_fg_dice")
          .round(4).to_string(index=False))

if df_segformer_cross is not None:
    print("\nSegFormer — cross-domain results (sorted by mean_fg_dice ascending)")
    print(df_segformer_cross[display_cols].sort_values("mean_fg_dice")
          .round(4).to_string(index=False))

if df_unet_cross is None and df_segformer_cross is None:
    print("No cross-domain data available. Update NEW_DATASET_PATH in Cell 2.")

## 12 · Save Cross-Domain Results

Persist per-subject DataFrames and the summary table to JSON/CSV for reproducibility.

In [ ]:
if df_unet_cross is not None:
    out = OUT_DIR / "cross_domain_unet_results.csv"
    df_unet_cross.to_csv(out, index=False)
    print(f"Saved → {out}")

if df_segformer_cross is not None:
    out = OUT_DIR / "cross_domain_segformer_results.csv"
    df_segformer_cross.to_csv(out, index=False)
    print(f"Saved → {out}")

if not summary_df.empty:
    out = OUT_DIR / "cross_domain_summary.csv"
    summary_df.to_csv(out, index=False)
    print(f"Saved → {out}")